## Hospital Sales Analysis — Manual DataFrame
## Data will include:

Products — Medical supplies, equipment, medicines\
Customers — Hospitals from our dataset\
Metrics — Quantity, Selling Price, Cost, Profit, Revenue

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *

In [0]:
%run "./01_connectivity"

Connectivity configured successfully!


In [0]:
# Manual sales data — hospital domain
data = [
    Row(sales_id=1,  product_name="Surgical Gloves",         customer_name="General Hospital",               quantity=500,  selling_price=2.50,   cost=1.20),
    Row(sales_id=2,  product_name="MRI Contrast Agent",      customer_name="Albert Einstein Medical Center", quantity=100,  selling_price=45.00,  cost=28.00),
    Row(sales_id=3,  product_name="IV Drip Set",             customer_name="Burn Brae Hospital",             quantity=300,  selling_price=8.75,   cost=4.50),
    Row(sales_id=4,  product_name="Surgical Gloves",         customer_name="All Saints Hospital",            quantity=750,  selling_price=2.50,   cost=1.20),
    Row(sales_id=5,  product_name="Ventilator",              customer_name="General Hospital",               quantity=5,    selling_price=8500.00,cost=6200.00),
    Row(sales_id=6,  product_name="Pulse Oximeter",          customer_name="Dufur Hospital",                 quantity=50,   selling_price=120.00, cost=75.00),
    Row(sales_id=7,  product_name="MRI Contrast Agent",      customer_name="Burn Brae Hospital",             quantity=80,   selling_price=45.00,  cost=28.00),
    Row(sales_id=8,  product_name="IV Drip Set",             customer_name="General Hospital",               quantity=600,  selling_price=8.75,   cost=4.50),
    Row(sales_id=9,  product_name="Surgical Mask",           customer_name="Albert Einstein Medical Center", quantity=2000, selling_price=0.75,   cost=0.30),
    Row(sales_id=10, product_name="Defibrillator",           customer_name="All Saints Hospital",            quantity=3,    selling_price=3200.00,cost=2400.00),
    Row(sales_id=11, product_name="Pulse Oximeter",          customer_name="General Hospital",               quantity=30,   selling_price=120.00, cost=75.00),
    Row(sales_id=12, product_name="Surgical Mask",           customer_name="Dufur Hospital",                 quantity=5000, selling_price=0.75,   cost=0.30),
    Row(sales_id=13, product_name="Ventilator",              customer_name="Albert Einstein Medical Center", quantity=3,    selling_price=8500.00,cost=6200.00),
    Row(sales_id=14, product_name="Defibrillator",           customer_name="Burn Brae Hospital",             quantity=2,    selling_price=3200.00,cost=2400.00),
    Row(sales_id=15, product_name="IV Drip Set",             customer_name="Dufur Hospital",                 quantity=400,  selling_price=8.75,   cost=4.50),
]

df_sales = spark.createDataFrame(data)
print(f"Sales records created: {df_sales.count()} rows")
df_sales.display()

Sales records created: 15 rows


sales_id,product_name,customer_name,quantity,selling_price,cost
1,Surgical Gloves,General Hospital,500,2.5,1.2
2,MRI Contrast Agent,Albert Einstein Medical Center,100,45.0,28.0
3,IV Drip Set,Burn Brae Hospital,300,8.75,4.5
4,Surgical Gloves,All Saints Hospital,750,2.5,1.2
5,Ventilator,General Hospital,5,8500.0,6200.0
6,Pulse Oximeter,Dufur Hospital,50,120.0,75.0
7,MRI Contrast Agent,Burn Brae Hospital,80,45.0,28.0
8,IV Drip Set,General Hospital,600,8.75,4.5
9,Surgical Mask,Albert Einstein Medical Center,2000,0.75,0.3
10,Defibrillator,All Saints Hospital,3,3200.0,2400.0


In [0]:
# Calculate Revenue, Cost and Profit

df_sales_analysis = df_sales \
    .withColumn("revenue", round(col("quantity") * col("selling_price"), 2)) \
    .withColumn("total_cost", round(col("quantity") * col("cost"), 2)) \
    .withColumn("profit", round(col("revenue") - col("total_cost"), 2)) \
    .withColumn("profit_margin_%", round((col("profit") / col("revenue")) * 100, 2))

print("Revenue, Cost, Profit calculated!")
df_sales_analysis.display()

Revenue, Cost, Profit calculated!


sales_id,product_name,customer_name,quantity,selling_price,cost,revenue,total_cost,profit,profit_margin_%
1,Surgical Gloves,General Hospital,500,2.5,1.2,1250.0,600.0,650.0,52.0
2,MRI Contrast Agent,Albert Einstein Medical Center,100,45.0,28.0,4500.0,2800.0,1700.0,37.78
3,IV Drip Set,Burn Brae Hospital,300,8.75,4.5,2625.0,1350.0,1275.0,48.57
4,Surgical Gloves,All Saints Hospital,750,2.5,1.2,1875.0,900.0,975.0,52.0
5,Ventilator,General Hospital,5,8500.0,6200.0,42500.0,31000.0,11500.0,27.06
6,Pulse Oximeter,Dufur Hospital,50,120.0,75.0,6000.0,3750.0,2250.0,37.5
7,MRI Contrast Agent,Burn Brae Hospital,80,45.0,28.0,3600.0,2240.0,1360.0,37.78
8,IV Drip Set,General Hospital,600,8.75,4.5,5250.0,2700.0,2550.0,48.57
9,Surgical Mask,Albert Einstein Medical Center,2000,0.75,0.3,1500.0,600.0,900.0,60.0
10,Defibrillator,All Saints Hospital,3,3200.0,2400.0,9600.0,7200.0,2400.0,25.0


In [0]:
# Highest cost per product

print("=== HIGHEST COST PER PRODUCT ===")
df_sales_analysis.groupBy("product_name") \
    .agg(
        round(sum("total_cost"), 2).alias("total_cost"),
        round(sum("revenue"), 2).alias("total_revenue"),
        round(sum("profit"), 2).alias("total_profit")
    ) \
    .orderBy("total_cost", ascending=False) \
    .display()

=== HIGHEST COST PER PRODUCT ===


product_name,total_cost,total_revenue,total_profit
Ventilator,49600.0,68000.0,18400.0
Defibrillator,12000.0,16000.0,4000.0
Pulse Oximeter,6000.0,9600.0,3600.0
IV Drip Set,5850.0,11375.0,5525.0
MRI Contrast Agent,5040.0,8100.0,3060.0
Surgical Mask,2100.0,5250.0,3150.0
Surgical Gloves,1500.0,3125.0,1625.0


In [0]:
# top buying customer

print("=== TOP BUYING CUSTOMER ===")
df_sales_analysis.groupBy("customer_name") \
    .agg(
        round(sum("revenue"), 2).alias("total_revenue"),
        round(sum("profit"), 2).alias("total_profit"),
        sum("quantity").alias("total_quantity")
    ) \
    .orderBy("total_revenue", ascending=False) \
    .display()

=== TOP BUYING CUSTOMER ===


customer_name,total_revenue,total_profit,total_quantity
General Hospital,52600.0,16050.0,1135
Albert Einstein Medical Center,31500.0,9500.0,2103
Dufur Hospital,13250.0,6200.0,5450
Burn Brae Hospital,12625.0,4235.0,382
All Saints Hospital,11475.0,3375.0,753


In [0]:
# Write sales analysis to Gold Delta Table

df_sales_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("databricks_generalhospital.gold.hospital_sales_gold")

print("Hospital Sales written to Gold Delta Table!")

Hospital Sales written to Gold Delta Table!
